# C1 (MB) — PC-MRI velocity profiles with corrected two-stage DRAM

**Reproducible example accompanying** *A Bayesian Framework for Uncertainty-Aware Estimation of Main Pulmonary Artery Velocity Profiles from Phase-Contrast MRI*.

This self-contained notebook reads the **magnitude DICOM**, **phase DICOM**, and **ImageJ ROI coordinates CSV**, performs ellipse fitting and 15-bin normalization, estimates observational uncertainty, and samples the Power-law and Womersley posteriors. Proposal tuning occurs only during burn-in; retained samples use a fixed proposal.

Outputs: separate MAP+UQ figures, marginal posterior densities (no mean/CI lines), trace plots, parameter summaries, and regional wRMSE. The analysis is for **MB / C1 (canine; blue observations)**.

**Reproducibility note:**  Posterior draws and MAP values will not be bit-for-bit identical to the original publication. No results are included until you run this notebook with the actual data.

In [ ]:
# If necessary, uncomment the next line in a Jupyter environment:
# %pip install numpy pandas scipy matplotlib pydicom arviz
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
from scipy.optimize import least_squares, minimize
from scipy.special import jv
from scipy.stats import gaussian_kde
import arviz as az
import pydicom

plt.rcParams.update({
    'font.family':'STIXGeneral', 'mathtext.fontset':'stix',
    'font.size':15, 'axes.labelsize':17, 'axes.titlesize':17,
    'xtick.labelsize':14, 'ytick.labelsize':14, 'legend.fontsize':12,
    'axes.linewidth':1.0, 'savefig.dpi':350,
})

SUBJECT = 'C1'
DATA_DIR = Path('MB')  # Or Path('.') if the three input files are beside the notebook
OUTPUT_DIR = Path('results_MB_C1')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MAG_FILE = DATA_DIR / 'IMG_0001_0002_m.dcm' # Magnitude image
PHASE_FILE = DATA_DIR / 'IMG_0001_0001_p.dcm' # Phase image
COORD_FILE = DATA_DIR / 'MB_COORDS.csv' # Dervied from ImageJ

SMOKE_TEST = False  # True: short debugging run; False: full inference
N_CHAINS = 2 if SMOKE_TEST else 4
N_ITER = 1500 if SMOKE_TEST else 60000
BURN_IN = 500 if SMOKE_TEST else 10000
N_BOOT = 100 if SMOKE_TEST else 1000
N_UQ = 200 if SMOKE_TEST else 3000
SEED = 42

# Original MB notebook: 0.5 s cycle, nu=2.91e-6 m²/s.
T_CYCLE_SECONDS = 0.5
KINEMATIC_VISCOSITY_M2_S = 2.91e-6
# Optional: supply the original full-precision alpha prior center, if documented.
# Keep None to calculate alpha_phys from the ellipse geometry and DICOM pixel spacing.
ALPHA_PRIOR_CENTER_OVERRIDE = None

for p in [MAG_FILE, PHASE_FILE, COORD_FILE]:
    if not p.is_file():
        raise FileNotFoundError(f'Missing input: {p}. See README.md for expected repository layout.')
print('Analysis:',SUBJECT,'| Full run:',not SMOKE_TEST)

## 1. Preprocessing functions

The geometry, binning, normalization, and model equations below were extracted from the supplied `MB_MCMC(4).ipynb`. The data-derived \(ar U\) calculation is deliberately preserved as implemented in that notebook to avoid silently changing the prior. See README for the distinction between this implementation and the manuscript's written area-integral formula.

In [ ]:
import numpy as np
from scipy.special import jv

def ellipse_radius(x, y, x0, y0, a, b, theta):
    ct = np.cos(theta)
    st = np.sin(theta)

    x_shift = x - x0
    y_shift = y - y0

    x_rot = x_shift * ct + y_shift * st
    y_rot = -x_shift * st + y_shift * ct

    r = np.sqrt((x_rot / a) ** 2 + (y_rot / b) ** 2)
    return r

def radial_bin_stats(r, I, num_bins):
    bins = np.linspace(0, 1.0, num_bins + 1)
    centers = 0.5 * (bins[:-1] + bins[1:])

    I_mean = np.full(num_bins, np.nan)
    I_std = np.full(num_bins, np.nan)
    counts = np.zeros(num_bins, dtype=int)

    for i in range(num_bins):
        m = (r >= bins[i]) & (r < bins[i + 1])
        counts[i] = np.sum(m)

        if counts[i] > 0:
            I_mean[i] = np.mean(I[m])
            I_std[i] = np.std(I[m])

    return centers, I_mean, I_std, counts

def robust_normalize_profile(r_vals, I_vals, center_thr=0.2, edge_thr=0.9):
    center_mask = r_vals < center_thr
    edge_mask = r_vals > edge_thr

    if np.sum(center_mask) == 0 or np.sum(edge_mask) == 0:
        raise ValueError("Not enough center or edge bins.")

    I_center = np.median(I_vals[center_mask])
    I_edge = np.median(I_vals[edge_mask])

    denom = I_center - I_edge

    if np.abs(denom) < 1e-12:
        raise ValueError("Center and edge values are too similar.")

    I_norm = (I_vals - I_edge) / denom
    I_norm = np.clip(I_norm, 0, 1)

    return I_norm, I_center, I_edge

def power_law_profile(r, gamma, Ubar):
    return Ubar * (gamma + 2) / gamma * (1 - r**gamma)

def compute_ubar_from_profile(r_vals, I_vals, a_fit, b_fit):
    area_cs = np.pi * a_fit * b_fit
    Q = 2 * np.pi * a_fit * b_fit * np.trapz(r_vals * I_vals, r_vals)
    Ubar = Q / area_cs
    return Ubar

def womersley_profile(r, A, phi, alpha, C=0.0):
    r = np.asarray(r, dtype=float)

    lam = (1j)**1.5 * alpha
    J0_lam = jv(0, lam)

    shape_complex = 1.0 - jv(0, lam * r) / J0_lam
    u = A * np.real(shape_complex * np.exp(1j * phi)) + C

    return u

def weighted_rmse(y, yhat, weights):
    y = np.asarray(y, dtype=float)
    yhat = np.asarray(yhat, dtype=float)
    weights = np.asarray(weights, dtype=float)

    return np.sqrt(np.sum(weights * (y - yhat)**2) / np.sum(weights))

In [ ]:
# Load ROI coordinates exported from ImageJ (first two columns: x, y pixel coordinates).
coords = pd.read_csv(COORD_FILE)
x = np.round(coords.iloc[:,0].to_numpy()).astype(int)
y = np.round(coords.iloc[:,1].to_numpy()).astype(int)
ds_mag = pydicom.dcmread(MAG_FILE)
ds_phase = pydicom.dcmread(PHASE_FILE)
img_mag = ds_mag.pixel_array.astype(float)
img_phase = ds_phase.pixel_array.astype(float)
if img_mag.shape != img_phase.shape:
    raise ValueError('Magnitude and phase image dimensions differ.')
if not np.all((x>=0)&(x<img_mag.shape[1])&(y>=0)&(y<img_mag.shape[0])):
    raise ValueError('ImageJ coordinates are outside the DICOM image bounds.')

# Ellipse fit: retain the original MB fit objective and bounds.
x0_init, y0_init = float(x.mean()), float(y.mean())
a_init, b_init = (x.max()-x.min())/2, (y.max()-y.min())/2
x_span,y_span = x.max()-x.min(), y.max()-y.min()
lower = np.array([x0_init-3,y0_init-3,0.7*x_span/2,0.7*y_span/2,-np.pi/4])
upper = np.array([x0_init+3,y0_init+3,1.3*x_span/2,1.3*y_span/2,np.pi/4])
p0=np.clip(np.array([x0_init,y0_init,a_init,b_init,0.0]),lower+1e-6,upper-1e-6)
def ellipse_residual(p):
    x0,y0,a,b,theta=p
    r=ellipse_radius(x,y,x0,y0,a,b,theta)
    return np.concatenate([r-0.75,[0.3*(a-a_init),0.3*(b-b_init)],0.5*np.maximum(r-1,0)])
fit=least_squares(ellipse_residual,p0,bounds=(lower,upper),method='trf',max_nfev=2000)
x0_fit,y0_fit,a_fit,b_fit,theta_fit=fit.x
r_all=ellipse_radius(x,y,*fit.x)
inside=r_all<=1.0
r_in=r_all[inside]
I_in=img_phase[y[inside],x[inside]]
edge=I_in[r_in>0.85]
edge_level=np.median(edge) if len(edge)>=5 else np.median(I_in)
I_in_corr=I_in-edge_level
center=I_in_corr[r_in<0.25]
if len(center)>=5 and np.median(center)<0:
    I_in_corr=-I_in_corr

# Subject-specific physiology-derived alpha from DICOM geometry.
spacing=np.asarray(ds_mag.PixelSpacing,dtype=float)
if not np.isclose(spacing[0],spacing[1]):
    print('WARNING: unequal DICOM pixel spacing; original MB code used PixelSpacing[0] for both axes.')
R_m=np.sqrt(a_fit*b_fit)*float(spacing[0])/1000
alpha_computed=R_m*np.sqrt((2*np.pi/T_CYCLE_SECONDS)/KINEMATIC_VISCOSITY_M2_S)
alpha_phys=float(alpha_computed if ALPHA_PRIOR_CENTER_OVERRIDE is None else ALPHA_PRIOR_CENTER_OVERRIDE)
print(f'Fitted axes (pixels): a={a_fit:.3f}, b={b_fit:.3f}')
print(f'Radius (m): {R_m:.6g}; computed alpha_phys: {alpha_computed:.5f}; prior center: {alpha_phys:.5f}')

fig,ax=plt.subplots(figsize=(6.5,6))
ax.imshow(img_mag,cmap='gray')
ax.scatter(x,y,s=2,color='deepskyblue',label='ImageJ ROI')
t=np.linspace(0,2*np.pi,500)
xe=a_fit*np.cos(t);ye=b_fit*np.sin(t)
ax.plot(x0_fit+xe*np.cos(theta_fit)-ye*np.sin(theta_fit),
        y0_fit+xe*np.sin(theta_fit)+ye*np.cos(theta_fit),color='red',lw=2,label='Fitted ellipse')
ax.legend(loc='best');ax.set_title('C1 · ROI and fitted ellipse');fig.tight_layout()
fig.savefig(OUTPUT_DIR/'C1_ROI_ellipse.pdf');plt.show()

In [ ]:
# Original 15 equal-width radial bins and center/wall normalization.
NUM_BINS=15
r_all_bins,raw_mean,raw_std,counts=radial_bin_stats(r_in,I_in_corr,NUM_BINS)
valid=np.isfinite(raw_mean)&(counts>0)
r_fit=r_all_bins[valid]
I_fit=raw_mean[valid]
I_std=raw_std[valid]
N_fit=counts[valid]
if len(r_fit)!=NUM_BINS:
    raise ValueError(f'Expected 15 populated radial bins; found {len(r_fit)}')
I_n,I_center_n,I_edge_n=robust_normalize_profile(r_fit,I_fit,center_thr=0.2,edge_thr=0.9)
den=I_center_n-I_edge_n
I_n_std=I_std/abs(den)
sigma_sem=np.maximum(I_n_std/np.sqrt(N_fit),1e-8)

rng_boot=np.random.default_rng(SEED)
bins=np.linspace(0,1,NUM_BINS+1)
boot=[]
for _ in range(N_BOOT):
    vals=np.full(NUM_BINS,np.nan)
    for j in range(NUM_BINS):
        m=(r_in>=bins[j])&(r_in<bins[j+1])
        if np.any(m):
            raw=I_in_corr[m]
            vals[j]=np.mean(rng_boot.choice(raw,size=len(raw),replace=True))
    if np.all(np.isfinite(vals[valid])):
        norm,_,_=robust_normalize_profile(r_fit,vals[valid],0.2,0.9)
        boot.append(norm)
if len(boot)<10:
    raise RuntimeError('Too few valid bootstrap realizations.')
sigma_norm=np.std(np.asarray(boot),axis=0)
sigma_data=np.maximum(np.sqrt(sigma_sem**2+sigma_norm**2),1e-8)
Ubar_data=float(compute_ubar_from_profile(r_fit,I_n,a_fit,b_fit))
print(f'Ubar_data = {Ubar_data:.6f}; alpha_phys = {alpha_phys:.6f}')
pd.DataFrame({'r':r_fit,'I_normalized':I_n,'sigma_sem':sigma_sem,
              'sigma_norm':sigma_norm,'sigma_data':sigma_data,'pixel_count':N_fit}).to_csv(OUTPUT_DIR/'C1_binned_data.csv',index=False)

## 2. Posterior definitions

Both likelihoods use \(\sigma_{\mathrm{obs},i}^2=\sigma_{\mathrm{data},i}^2+\sigma_{\mathrm{model}}^2\). Sampling is performed in **log sigma** coordinates with the Jacobian included. Priors follow the original MB notebook, including its model-specific half-normal scales (0.05 for Power-law, 0.03 for Womersley).

In [ ]:
def log_posterior_power(theta):
    gamma,ubar,logsigma=np.asarray(theta,dtype=float)
    if not (2<=gamma<=9 and 0.5*Ubar_data<=ubar<=1.5*Ubar_data and np.log(1e-5)<logsigma<np.log(0.15)):
        return -np.inf
    sig=np.exp(logsigma)
    pred=power_law_profile(r_fit,gamma,ubar)
    s=np.sqrt(sigma_data**2+sig**2)
    resid=(I_n-pred)/s
    return float(-0.5*np.sum(resid**2+np.log(2*np.pi*s**2))
                 -0.5*((ubar-Ubar_data)/(0.25*Ubar_data))**2
                 -0.5*(sig/0.05)**2+logsigma)

def log_posterior_wom(theta):
    A,phi,alpha,logsigma=np.asarray(theta,dtype=float)
    if not (A>0 and -np.pi<=phi<=np.pi and alpha>1e-3 and np.log(1e-5)<logsigma<np.log(0.15)):
        return -np.inf
    sig=np.exp(logsigma)
    pred=womersley_profile(r_fit,A,phi,alpha,C=0)
    if not np.all(np.isfinite(pred)):
        return -np.inf
    s=np.sqrt(sigma_data**2+sig**2)
    resid=(I_n-pred)/s
    return float(-0.5*np.sum(resid**2+np.log(2*np.pi*s**2))
                 -0.5*((alpha-alpha_phys)/2.0)**2
                 -0.5*(sig/0.03)**2+logsigma)

## 3. DRAM

Stage 1: \(y\sim N(x,C_1)\). On rejection, stage 2: \(z\sim N(x,C_2)\), where \(C_2=s^2C_1\). The second-stage acceptance probability includes **both** the forward/reverse first-stage proposal densities and rejection probabilities. The second-stage proposal density cancels because it is a symmetric random walk independent of the rejected point. The proposal scale is tuned **only during burn-in**.

The proposal covariance is initialized using a regularized numerical curvature approximation near a posterior mode. The original, nonstandard sampler is **not** used.

In [ ]:
from scipy.special import logsumexp

def _log_gaussian_kernel(delta, chol):
    """Log multivariate normal density excluding shared normalizer."""
    v = np.linalg.solve(chol, delta)
    return -0.5 * float(v @ v)


def _log_rejection_probability(log_alpha_ratio):
    """log[1 - min(1, exp(log_alpha_ratio))], numerically stable."""
    if log_alpha_ratio >= 0.0:
        return -np.inf
    return float(np.log(-np.expm1(log_alpha_ratio)))


def run_correct_dram(log_posterior, initial, pilot_cov, *,
                     n_iter=60000, burn_in=10000, seed=1234,
                     delayed_scale=0.10, target_acceptance=0.28,
                     adapt_interval=200):
    """Sample a log posterior using correct two-stage delayed rejection.

    Stage 1: y ~ N(x, C1).
    Stage 2: z ~ N(x, C2), conditional on rejection of y.

    For q2(z|x,y)=N(z;x,C2), q2(x|z,y)=N(x;z,C2), so their
    densities cancel in the delayed-rejection ratio. The q1 densities
    generally do NOT cancel, even though q1 is a symmetric random walk.

    Returns:
        samples: shape (n_iter - burn_in, n_parameters)
        log_probabilities: shape (n_iter - burn_in,)
        info: stage-specific acceptance fractions over retained iterations
    """
    if not (0 < burn_in < n_iter):
        raise ValueError('Require 0 < burn_in < n_iter')
    if not (0 < delayed_scale < 1):
        raise ValueError('Require 0 < delayed_scale < 1')
    rng = np.random.default_rng(seed)
    x = np.asarray(initial, dtype=float).copy()
    if x.ndim != 1:
        raise ValueError('initial must be one-dimensional')
    d = len(x)
    cov = np.asarray(pilot_cov, dtype=float)
    if cov.shape != (d, d):
        raise ValueError('pilot_cov must have shape (d, d)')
    L = np.linalg.cholesky(cov)
    lx = float(log_posterior(x))
    if not np.isfinite(lx):
        raise ValueError('initial has nonfinite log posterior')
    log_scale = float(np.log(2.38 / np.sqrt(d)))
    samples = np.empty((n_iter - burn_in, d))
    log_probs = np.empty(n_iter - burn_in)
    n_first = n_second = 0
    accepted_in_window = 0
    adaptation_steps = 0

    for t in range(n_iter):
        L1 = np.exp(log_scale) * L
        L2 = delayed_scale * L1
        y = x + L1 @ rng.standard_normal(d)
        ly = float(log_posterior(y))
        lr1 = ly - lx if np.isfinite(ly) else -np.inf
        accept1 = np.isfinite(ly) and np.log(rng.random()) < min(0.0, lr1)
        accept2 = False
        if accept1:
            x, lx = y, ly
        else:
            z = x + L2 @ rng.standard_normal(d)
            lz = float(log_posterior(z))
            if np.isfinite(lz):
                # Reverse first-stage acceptance probability alpha1(z,y).
                lr1_reverse = ly - lz if np.isfinite(ly) else -np.inf
                log_reject_forward = _log_rejection_probability(lr1)
                log_reject_reverse = _log_rejection_probability(lr1_reverse)
                if np.isfinite(log_reject_forward) and np.isfinite(log_reject_reverse):
                    log_q1_ratio = (_log_gaussian_kernel(y - z, L1)
                                    - _log_gaussian_kernel(y - x, L1))
                    log_ratio2 = (lz - lx + log_q1_ratio
                                  + log_reject_reverse - log_reject_forward)
                    accept2 = np.log(rng.random()) < min(0.0, log_ratio2)
                    if accept2:
                        x, lx = z, lz

        if t < burn_in:
            accepted_in_window += int(accept1 or accept2)
            if (t + 1) % adapt_interval == 0:
                adaptation_steps += 1
                learning_rate = min(0.5, 1.0 / np.sqrt(adaptation_steps))
                log_scale += learning_rate * (
                    accepted_in_window / adapt_interval - target_acceptance)
                log_scale = float(np.clip(log_scale, np.log(0.03), np.log(50.0)))
                accepted_in_window = 0
        else:
            i = t - burn_in
            samples[i] = x
            log_probs[i] = lx
            n_first += int(accept1)
            n_second += int(accept2)

    n_saved = n_iter - burn_in
    return samples, log_probs, {
        'first_acceptance': n_first / n_saved,
        'second_acceptance': n_second / n_saved,
        'total_acceptance': (n_first + n_second) / n_saved,
        'frozen_scale': float(np.exp(log_scale)),
    }

In [ ]:
# Estimate a starting covariance using finite-difference curvature at a fitted mode.
# The covariance only affects efficiency, not the posterior target.
def fit_mode_and_cov(logp,guess,scales,bounds):
    guess=np.asarray(guess,dtype=float)
    scales=np.asarray(scales,dtype=float)
    def objective(u):
        val=logp(guess+u*scales)
        return -val if np.isfinite(val) else 1e12
    opt=minimize(objective,np.zeros(len(guess)),method='Nelder-Mead',
                 options={'maxiter':5000,'xatol':1e-6,'fatol':1e-6})
    mode=guess+opt.x*scales
    if not np.isfinite(logp(mode)):
        raise RuntimeError('Unable to find finite posterior mode.')
    d=len(mode)
    # Numerical Hessian of negative log posterior in standardized coordinates.
    h=np.maximum(1e-3,np.abs(opt.x)*0+0.01)
    H=np.zeros((d,d));f0=objective(opt.x)
    for i in range(d):
        ei=np.zeros(d);ei[i]=h[i]
        H[i,i]=(objective(opt.x+ei)-2*f0+objective(opt.x-ei))/h[i]**2
        for j in range(i):
            ej=np.zeros(d);ej[j]=h[j]
            H[i,j]=H[j,i]=(objective(opt.x+ei+ej)-objective(opt.x+ei-ej)
                         -objective(opt.x-ei+ej)+objective(opt.x-ei-ej))/(4*h[i]*h[j])
    H=0.5*(H+H.T)
    vals,vecs=np.linalg.eigh(H)
    vals=np.clip(vals,0.01,1e5)
    cov_u=(vecs/vals)@vecs.T
    cov=np.diag(scales)@cov_u@np.diag(scales)
    # Small shrinkage ensures stable positive definiteness.
    cov=0.95*cov+0.05*np.diag(np.diag(cov))
    cov+=np.eye(d)*1e-10
    return mode,cov

# Use stable starting guesses based on the original parameterization.
models={
    'Power-law':{
        'logp':log_posterior_power,
        'initial':np.array([6.0,Ubar_data,np.log(0.03)]),
        'scales':np.array([1.0,max(0.05,Ubar_data*0.1),0.5]),
        'labels':['gamma','Ubar','sigma_model'],
    },
    'Womersley':{
        'logp':log_posterior_wom,
        'initial':np.array([1.8,1.0,alpha_phys,np.log(0.03)]),
        'scales':np.array([0.3,0.15,1.5,0.5]),
        'labels':['A','phi','alpha','sigma_model'],
    }
}
for name,info in models.items():
    mode,cov=fit_mode_and_cov(info['logp'],info['initial'],info['scales'],None)
    info['mode'],info['pilot_cov']=mode,cov
    print(name,'pilot mode:',mode)

In [ ]:
# Run corrected DRAM independently for both models and save every retained chain.
results={}
for m_index,(model,info) in enumerate(models.items()):
    print('\n',model,'— running',N_CHAINS,'chains')
    chains=[]; logps=[]; diagnostics=[]
    for c in range(N_CHAINS):
        rng=np.random.default_rng(SEED+1000*m_index+c)
        x0=info['mode'].copy()
        for attempt in range(100):
            proposal=info['mode']+0.12*np.linalg.cholesky(info['pilot_cov'])@rng.normal(size=len(x0))
            if np.isfinite(info['logp'](proposal)):
                x0=proposal;break
        draw,lp,diag=run_correct_dram(
            info['logp'],x0,info['pilot_cov'],n_iter=N_ITER,burn_in=BURN_IN,
            seed=SEED+1000*m_index+100*c,delayed_scale=0.10)
        chains.append(draw);logps.append(lp);diagnostics.append(diag)
        print(f'  chain {c+1}: first={diag["first_acceptance"]:.3f}, second={diag["second_acceptance"]:.3f}')
    chains=np.asarray(chains);logps=np.asarray(logps)
    np.savez_compressed(OUTPUT_DIR/f'C1_{model.replace("-", "_")}_chains.npz',chains=chains,log_posterior=logps)
    results[model]={'chains':chains,'logps':logps,'diagnostics':diagnostics}

print('Sampling complete.')

## 4. Diagnostics and posterior summaries

The MAP is the retained draw with the highest evaluated **transformed-coordinate** log posterior, matching the original notebook's sample-MAP convention. This is not identical to optimizing the density in physical sigma coordinates.

**Do not use smoke-test draws for publication.** Review all R-hat and ESS values before using posterior summaries.

In [ ]:
summary_rows=[];diag_rows=[]
for model,info in models.items():
    r=results[model]
    chains=r['chains'].copy()
    physical=chains.copy()
    physical[:,:,-1]=np.exp(physical[:,:,-1])
    pooled=physical.reshape(-1,physical.shape[-1])
    flat_lp=r['logps'].reshape(-1)
    map_index=int(np.argmax(flat_lp))
    r['map']=pooled[map_index]
    r['physical_chains']=physical
    r['pooled']=pooled
    for j,name in enumerate(info['labels']):
        vals=physical[:,:,j]
        rhat=float(az.rhat(az.convert_to_dataset({'v':vals}))['v'].values)
        ess=float(az.ess(az.convert_to_dataset({'v':vals}),method='bulk')['v'].values)
        summary_rows.append({'subject':'C1','model':model,'parameter':name,
            'MAP':r['map'][j],'mean':pooled[:,j].mean(),'median':np.median(pooled[:,j]),
            'CI_2.5':np.percentile(pooled[:,j],2.5),'CI_97.5':np.percentile(pooled[:,j],97.5),
            'Rhat':rhat,'ESS_bulk':ess})
        diag_rows.append({'subject':'C1','model':model,'parameter':name,'Rhat':rhat,'ESS_bulk':ess})
summary=pd.DataFrame(summary_rows)
summary.to_csv(OUTPUT_DIR/'C1_posterior_summary.csv',index=False)
pd.DataFrame(diag_rows).to_csv(OUTPUT_DIR/'C1_convergence_diagnostics.csv',index=False)
print(summary.to_string(index=False,float_format=lambda v:f'{v:.5g}'))
if (summary.Rhat>1.01).any() or (summary.ESS_bulk<400).any():
    print('WARNING: At least one parameter has Rhat > 1.01 or ESS < 400. Review sampling before publication.')

In [ ]:
# Separate marginal posterior density plots (no posterior mean or CI markers).
for model,info in models.items():
    r=results[model]
    for j,name in enumerate(info['labels']):
        vals=r['pooled'][:,j]
        xx=np.linspace(np.percentile(vals,0.1),np.percentile(vals,99.9),400)
        fig,ax=plt.subplots(figsize=(7,5))
        ax.plot(xx,gaussian_kde(vals)(xx),color='#2166ac',lw=2.5)
        ax.set_xlabel(name);ax.set_ylabel('Posterior density')
        ax.grid(alpha=0.15);fig.tight_layout()
        stem=f'C1_{model.replace("-", "_")}_posterior_{name}'
        fig.savefig(OUTPUT_DIR/f'{stem}.pdf',bbox_inches='tight')
        fig.savefig(OUTPUT_DIR/f'{stem}.png',bbox_inches='tight')
        plt.close(fig)
    fig,axes=plt.subplots(len(info['labels']),1,figsize=(10,2.4*len(info['labels'])),sharex=True)
    for j,ax in enumerate(np.atleast_1d(axes)):
        for c in range(N_CHAINS):
            ax.plot(r['physical_chains'][c,:,j],lw=0.35,alpha=0.7)
        ax.set_ylabel(info['labels'][j])
    axes[-1].set_xlabel('Retained iteration');fig.tight_layout()
    fig.savefig(OUTPUT_DIR/f'C1_{model.replace("-", "_")}_trace.pdf',bbox_inches='tight')
    plt.close(fig)
print('Posterior density and trace figures saved.')

## 5. MAP fits, credible intervals, and prediction intervals

The model-function credible band is calculated from 3,000 retained posterior draws. The prediction interval additionally includes heteroscedastic data uncertainty and sampled residual model discrepancy. Model curves are evaluated over the **entire normalized radius [0, 1]**, including the exact no-slip wall. The observational prediction interval need not collapse at the wall.

In [ ]:
r_plot=np.linspace(0.0,1.0,501)
rng_pred=np.random.default_rng(SEED+999)
all_fit=[]
for model,info in models.items():
    r=results[model]
    pool=r['pooled'];n_draw=min(N_UQ,len(pool))
    chosen=pool[rng_pred.choice(len(pool),size=n_draw,replace=False)]
    if model=='Power-law':
        model_fn=lambda th,rad:power_law_profile(rad,th[0],th[1])
    else:
        model_fn=lambda th,rad:womersley_profile(rad,th[0],th[1],th[2],C=0)
    curve=np.array([model_fn(th,r_plot) for th in chosen])
    map_curve=model_fn(r['map'],r_plot)
    assert np.isclose(map_curve[-1],0,atol=1e-9),f'{model}: wall condition violated'
    assert np.max(np.abs(curve[:,-1]))<1e-8
    sig_interp=np.interp(r_plot,r_fit,sigma_data)
    noise=rng_pred.normal(size=curve.shape)*np.sqrt(sig_interp[None,:]**2+chosen[:,-1,None]**2)
    pred=curve+noise
    cl,ch=np.percentile(curve,[2.5,97.5],axis=0)
    pl,ph=np.percentile(pred,[2.5,97.5],axis=0)
    fit_bins=model_fn(r['map'],r_fit)
    wrmse=weighted_rmse(I_n,fit_bins,N_fit)
    r['wRMSE']=wrmse
    all_fit.append({'subject':'C1','model':model,'wRMSE':wrmse,'RMSE':np.sqrt(np.mean((I_n-fit_bins)**2))})
    pd.DataFrame({'r':r_plot,'MAP':map_curve,'CI_lower':cl,'CI_upper':ch,'PI_lower':pl,'PI_upper':ph}).to_csv(
        OUTPUT_DIR/f'C1_{model.replace("-", "_")}_UQ_curves.csv',index=False)
    fig,ax=plt.subplots(figsize=(8,6))
    ax.fill_between(r_plot,pl,ph,color='lightgray',label='95% prediction interval',zorder=1)
    ax.fill_between(r_plot,cl,ch,color='gray',alpha=0.85,label='95% credible interval',zorder=2)
    ax.plot(r_plot,map_curve,'k-',lw=2,label='MAP prediction',zorder=3)
    ax.errorbar(r_fit,I_n,yerr=sigma_data,fmt='o',color='#2166ac',ms=4,
                capsize=2,elinewidth=1,label='Data',zorder=4)
    ax.set_xlim(0,1);ax.margins(x=0)
    low=min(np.min(pl),np.min(cl),np.min(I_n-sigma_data),np.min(map_curve))
    high=max(np.max(ph),np.max(ch),np.max(I_n+sigma_data),np.max(map_curve))
    pad=0.05*(high-low)
    ax.set_ylim(low-pad,high+pad)
    ax.set_xlabel('Normalized elliptical radius $r$')
    ax.set_ylabel('Normalized phase / velocity')
    ax.set_title(f'C1 · {model}')
    ax.legend(loc='lower left',frameon=False)
    ax.grid(alpha=0.16);fig.tight_layout()
    stem=f'C1_{model.replace("-", "_")}_fit_with_UQ'
    fig.savefig(OUTPUT_DIR/f'{stem}.pdf',bbox_inches='tight')
    fig.savefig(OUTPUT_DIR/f'{stem}.png',bbox_inches='tight')
    plt.show()
pd.DataFrame(all_fit).to_csv(OUTPUT_DIR/'C1_fit_metrics.csv',index=False)

In [ ]:
# Regional weighted RMSE for the same bin-index regions used in the manuscript.
# Center: first 9; transitional: next 3; wall: last 3 (15 bins total).
regions={'Global':np.arange(15),'Center':np.arange(9),
         'Transitional':np.arange(9,12),'Wall':np.arange(12,15)}
rows=[]
for model in models:
    th=results[model]['map']
    yhat=(power_law_profile(r_fit,th[0],th[1]) if model=='Power-law'
          else womersley_profile(r_fit,th[0],th[1],th[2]))
    for region,ii in regions.items():
        rows.append({'subject':'C1','model':model,'region':region,
                     'wRMSE':weighted_rmse(I_n[ii],yhat[ii],N_fit[ii])})
pd.DataFrame(rows).to_csv(OUTPUT_DIR/'C1_regional_wRMSE.csv',index=False)

# Compact Table 2 row (posterior MAP values, plus prior centers derived before sampling).
p=results['Power-law']['map'];w=results['Womersley']['map']
row=pd.DataFrame([{'ID':'C1','gamma_MAP':p[0],'Ubar_MAP':p[1],'Ubar_data':Ubar_data,
    'alpha_MAP':w[2],'alpha_phys':alpha_phys,'A_MAP':w[0],'phi_MAP':w[1]}])
row.to_csv(OUTPUT_DIR/'C1_Table2_row.csv',index=False)
print(row.round(4).to_string(index=False))
print('All files saved to:',OUTPUT_DIR.resolve())